# Data Preprocessing & Feature Engineering

It is the main nootebook related to Data Preprocessing & Feature Engineering of https://www.kaggle.com/datasets/sahistapatel96/bankadditionalfullcsv dataset.

Previous notbook related to EDA - https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/01_eda.ipynb




## Notebook initialization

In [1]:
# Notebook initialization
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# print("Project root:", ROOT)

## Imports & Load data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from src.data import load_raw_data

raw_df = load_raw_data()
raw_df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
target_col = "y"

## Split Data Train & Validation

Further during modeling, cross-validation can be used.

Here, the dataset is split into training and validation sets with an 80/20 ratio (4:1). The split is stratified by the target variable to preserve the class distribution due to target class imbalance.

In [4]:
from src.data import split_train_val

train_df, val_df = split_train_val(raw_df, stratify_col=target_col)
train_df.shape, val_df.shape

((32950, 21), (8238, 21))

In [5]:
train_df[target_col].value_counts(normalize=True), val_df[target_col].value_counts(normalize=True)

(y
 no     0.887344
 yes    0.112656
 Name: proportion, dtype: float64,
 y
 no     0.887351
 yes    0.112649
 Name: proportion, dtype: float64)

## Custom Numeric Column Transformers

The plan is to use scikit-learn Pipelines for modeling.
To keep the logic simple and readable, we implement a few custom transformers to be reused in future pipeline experiments.


### Domain Specific Column Transformers

Column Transformers specific for <a href="https://www.kaggle.com/datasets/sahistapatel96/bankadditionalfullcsv">bank additional</a> dataset

#### CampaignPrevTransformer
Creates `campaign_prev = min(campaign - 1, 5)`

In [6]:
from src.preprocessing import CampaignPrevTransformer
campaign_tr = CampaignPrevTransformer()

test_df = campaign_tr.fit_transform(train_df)

test_df[test_df["campaign"] > 4][["campaign", "campaign_prev"]].head(6)

,campaign,campaign_prev
14697,5,4
17537,5,4
1595,5,4
4216,5,4
6347,9,5
9348,7,5


In [7]:
test_df["campaign_prev"].value_counts().sort_index()

campaign_prev
0    14121
1     8469
2     4300
3     2116
4     1255
5     2689
Name: count, dtype: int64

#### PdaysTransformer

Depending on the `mode` parameter, this transformer converts `pdays` into:

 - a **binary feature**: `pdays_contacted = 1` if `pdays != 999`, otherwise `0`
 - **categorical groups**, such as:
   - `not_contacted_before`
   - `contacted_recently` (e.g., `< 14` days)
   - `contacted_long_ago` (e.g., `≥ 14` days)

In [8]:
from src.preprocessing import PdaysTransformer

# binary case
pdays_tr = PdaysTransformer(mode="binary")
df_test = pdays_tr.fit_transform(train_df)
df_test[["pdays", "pdays_contacted"]].iloc[-10:-5]

,pdays,pdays_contacted
15913,999,0
30274,2,1
24607,999,0
27529,999,0
34253,999,0


In [9]:
# group case
pdays_tr = PdaysTransformer(mode="group", recent_days=7)
df_test = pdays_tr.fit_transform(train_df)
df_test[df_test["pdays"] < 999][["pdays", "pdays_group"]].head()

,pdays,pdays_group
39226,3,contacted_recently
39224,6,contacted_recently
40273,3,contacted_recently
35478,10,contacted_long_ago
39714,8,contacted_long_ago


### Generic Transformers

Custom column transformers that are **not dataset-specific** and can be **reused with any dataset**.

#### ColumnDropper

This step can be used:
 - as the **first step** to **drop features that will not be used in the model** at an early stage
 - to **remove columns that have already been preprocessed** and are **no longer relevant for modeling**

In [10]:
from src.preprocessing import ColumnDropper

cols_to_drop = ["duration", "contact", "housing", "loan"]

pipe_auto = make_pipeline(ColumnDropper(columns=cols_to_drop))

train_drop_df = pipe_auto.fit_transform(train_df)

print("Original columns:", train_df.shape[1])
print("New columns:", train_drop_df.shape[1])
print("Dropped columns:", set(train_df.columns) - set(train_drop_df.columns))

Original columns: 21
New columns: 17
Dropped columns: {'housing', 'contact', 'loan', 'duration'}


#### OutlierCapper

In [11]:
from src.preprocessing import OutlierCapper

# Cap campaign at 6 explicitly
capper = OutlierCapper(column="campaign", cap=6)

train_transformed = capper.fit_transform(train_df)

# Check results
train_transformed["campaign"].value_counts().sort_index()

campaign
1    14121
2     8469
3     4300
4     2116
5     1255
6     2689
Name: count, dtype: int64

In [12]:
# Cap campaign at 6 implicitly using iqr otliers method
capper_iqr = OutlierCapper(column="campaign")
train_transformed_iqr = capper_iqr.fit_transform(train_df)
train_transformed_iqr["campaign"].value_counts().sort_index()

campaign
1.0    14121
2.0     8469
3.0     4300
4.0     2116
5.0     1255
6.0     2689
Name: count, dtype: int64

#### ColumnArithmetic

- Applies an arithmetic operation (**add, subtract, multiply, divide**) to a column.
- When used together with **OutlierCapper**, it can reproduce the same transformation as the domain-specific `CampaignPrevTransformer`.
- Since feature scaling may be applied at later preprocessing stages, `ColumnArithmetic` may be **redundant for this specific `campaign` case**.

In [13]:
from src.preprocessing import ColumnArithmetic

pipe = make_pipeline(
    ColumnArithmetic(column="campaign", operation="subtract", value=1),
    OutlierCapper(column="campaign", cap=5),
)

train_transformed = pipe.fit_transform(train_df)
train_transformed["campaign"].value_counts().sort_index()

campaign
0    14121
1     8469
2     4300
3     2116
4     1255
5     2689
Name: count, dtype: int64

#### NumericBinner

Transforms a numeric column into categorical bins based on specified thresholds.

In [ ]:
from src.preprocessing import NumericBinner

# example for age column
age_binner = NumericBinner(
    column="age",
    bins=[0, 25, 60, 120],
    labels=["young", "middle", "old"]
)

train_age = age_binner.fit_transform(train_df)

train_age[["age", "age_group"]].head(5)

,age,age_group
25611,49,middle
26010,37,middle
40194,78,old
297,36,middle
36344,59,middle


In [ ]:
# can be used for pdays column too, but maybe not the best approach to handle this special 999 value
pdays_binner = NumericBinner(
    column="pdays",
    bins=[0, 7, 998, 1000],
    labels=["contacted_recently", "contacted_long_ago", "not_contacted_before"]
)

train_pdays = pdays_binner.fit_transform(train_df)
train_pdays[train_pdays["pdays"] < 999][["pdays", "pdays_group"]].head()

,pdays,pdays_group
39226,3,contacted_recently
39224,6,contacted_recently
40273,3,contacted_recently
35478,10,contacted_long_ago
39714,8,contacted_long_ago


#### ConditionalMapper

Applies rule-based mapping to create a new column

In [ ]:
from src.preprocessing import ConditionalMapper

# better approach for pdays column handling than above NumericBinner
mapper_pdays = ConditionalMapper(
    column="pdays",
    new_column="pdays_group",
    rules=[
        (lambda x: x == 999, "not_contacted_before"),
        (lambda x: (x < 14) & (x != 999), "contacted_recently"),
        (lambda x: x >= 14, "contacted_long_ago"),
    ]
)

df_pdays = mapper_pdays.fit_transform(train_df)
df_pdays[df_pdays["pdays"] < 999][["pdays", "pdays_group"]].head()

,pdays,pdays_group
39226,3,contacted_recently
39224,6,contacted_recently
40273,3,contacted_recently
35478,10,contacted_recently
39714,8,contacted_recently


#### CyclicalEncoder

Encode cyclical features using **sine and cosine transformations**.

Useful for: month, day_of_week, hour, etc.

**Note:** Sine alone is not sufficient to uniquely represent each value in the cycle because of periodic symmetry. Using **both sine and cosine** ensures that each value has a **unique representation** and preserves the cyclical relationship.

In [ ]:
from src.preprocessing.mappings import month_map, dow_map

train_df["month_num"] = train_df["month"].map(month_map)
train_df["day_of_week_num"] = train_df["day_of_week"].map(dow_map)

In [30]:
from src.preprocessing import CyclicalEncoder

# Month (1–12)
month_encoder = CyclicalEncoder(column="month_num", max_value=12)
train_df = month_encoder.fit_transform(train_df)

# Day of week (assuming Monday=0, Sunday=6)
dow_encoder = CyclicalEncoder(column="day_of_week_num", max_value=4)
train_df = dow_encoder.fit_transform(train_df)

# Check
train_df.head()[["month", "month_num_sin", "month_num_cos", "day_of_week", "day_of_week_num_sin", "day_of_week_num_cos"]]

,month,month_num_sin,month_num_cos,day_of_week,day_of_week_num_sin,day_of_week_num_cos
25611,nov,-5.000000e-01,0.866025,wed,1.224647e-16,-1.000000e+00
26010,nov,-5.000000e-01,0.866025,wed,1.224647e-16,-1.000000e+00
40194,jul,-5.000000e-01,-0.866025,mon,0.000000e+00,1.000000e+00
297,may,5.000000e-01,-0.866025,mon,0.000000e+00,1.000000e+00
36344,jun,1.224647e-16,-1.000000,tue,1.000000e+00,6.123234e-17


## Custom Categorical column Transformers
TODO